# Practice: Extract an Entity Type With No Training Data

**Computational Text Analysis Pack · Module 15 — Rule-Based and Symbolic Methods**

*Technique:* Building, measuring and iterating a rule set against a hand-checked sample

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/04-rule-based-methods/15e_practice_rules_vs_stats.ipynb)

Data: `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## The question

> **You need every mention of a specific kind of thing in an archive, you have
> no labelled data, and you cannot afford to annotate. Can rules get you there,
> and how would you know?**

The honest workflow: write a rule, check a sample by hand, measure, improve,
repeat. This is what module 15c's gazetteers look like as a research process
rather than a demonstration.

### How to use this notebook

A worked case, not a tutorial. Run it once for the shape of the argument, then
change the question at the top to one of your own.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
import re
import numpy as np
TARGET = "expedition and travel events"     # what we are trying to find
docs = df_colonial["text"].head(1200).astype(str).tolist()

ROUNDS = {}
ROUNDS["v1 bare keyword"] = r"\b(expedition|journey|voyage)\b"
ROUNDS["v2 + verb frame"] = r"\b(?:set out|departed|embarked|started)\s+(?:on|for)\s+[a-z]+"
ROUNDS["v3 + date anchor"] = r"\b(?:in|on)\s+1[6-9]\d{2}[^.]{0,60}\b(expedition|journey|voyage|march)\b"

for name, pat in ROUNDS.items():
    hits = sum(len(re.findall(pat, d, flags=re.I)) for d in docs)
    print(f"{name:<22}{hits:>7} matches")
print()
print("More matches is not better. The next cell is the only one that matters.")

In [ ]:
# Hand-check a sample. Twenty is enough to see the error classes.
rng = np.random.default_rng(0)
pat = ROUNDS["v1 bare keyword"]
found = []
for d in docs:
    for m in re.finditer(pat, d, flags=re.I):
        s, e = max(0, m.start()-70), min(len(d), m.end()+70)
        found.append(d[s:e].replace("\n", " "))
sample = [found[i] for i in rng.choice(len(found), size=min(12, len(found)), replace=False)]
print(f"{len(found)} total matches; a sample of {len(sample)} to check by eye:\n")
for i, s in enumerate(sample, 1):
    print(f"{i:>3}. ...{' '.join(s.split())}...")
print()
print("Score these yourself. Which are genuine references to a travel event, and")
print("which are metaphorical ('the journey of life'), or a book title, or a")
print("mention of somebody else's expedition?")

In [ ]:
# Suppose your hand-scoring gave these. Replace with your own numbers.
HAND_SCORED = {"v1 bare keyword": {"checked": 12, "correct": 7},
               "v2 + verb frame": {"checked": 12, "correct": 10},
               "v3 + date anchor": {"checked": 12, "correct": 11}}
rows = []
for name, pat in ROUNDS.items():
    total = sum(len(re.findall(pat, d, flags=re.I)) for d in docs)
    hs = HAND_SCORED[name]
    prec = hs["correct"] / hs["checked"]
    rows.append({"rule": name, "matches": total, "sampled_precision": prec,
                 "est_correct": int(total * prec)})
r = pd.DataFrame(rows)
print(r.to_string(index=False, float_format=lambda v: f"{v:.2f}"))
print()
print("The estimated-correct column is the one to optimise. v1 finds most and is")
print("least precise; v3 is precise and finds least. Which you want depends on")
print("whether a missed mention or a false one costs you more.")

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.bar(r["rule"], r["matches"], color="lightgrey", label="all matches")
ax.bar(r["rule"], r["est_correct"], color="steelblue", label="estimated correct")
ax.set_ylabel("mentions"); ax.legend()
ax.set_xticklabels(r["rule"], rotation=15, ha="right", fontsize=8)
ax.set_title("Precision is bought with recall")
plt.tight_layout()

## What this licenses

**Defensible:** "Mentions were extracted with the pattern [exact regex]. A
random sample of 12 matches was hand-checked, giving an estimated precision of
[X]; the pattern was iterated three times, and the rules are published with the
data."

That paragraph gives a reader everything: the instrument, its measured error,
and the ability to rerun it. It is a stronger methods statement than most
machine-learning papers manage, and it took an afternoon and no training data.

**Not defensible:** a count of matches. Without the hand-checked precision, "412
mentions of expeditions" is a count of regex hits, not of expeditions.

**And the limit from module 15c:** these rules encode English word order and
capitalisation. They will not transfer to another language, and saying so is
part of reporting them.

### 🔧 Your turn 1

Write a fourth rule for the same target that trades the other way — maximum recall, low precision — and hand-check twelve of its matches. Which rule would you actually use?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

The answer usually depends on what comes next. If a human will read every match,
high recall wins — you can discard false positives by eye but cannot recover
what was never found. If the matches feed a count, high precision wins, because
every false positive becomes a wrong number. Deciding this *before* writing the
rule, rather than choosing whichever rule gave the nicer figure, is the
difference between an instrument and a rationalisation.

</details>